# Shipd - Weather Translation Challenge
Sequence-to-sequence translation of 72-hour weather observations between weather stations, trained from scratch (no pretrained weather models or external forecast data permitted).

**Architecture.** BiLSTM encoder with per-station embeddings operating in *climate-anomaly* space â€” each sample's source series is first centred on the source station's learned climate mean, the network predicts the target-station anomaly, then the target climate mean is added back. This factorises prediction into (a) an exact climate offset derived from the station's own history and (b) a learned station-pair anomaly transformation. It substantially helps the model generalise to the two station pairs present only in the test set.

**Ensemble.** Multiple LSTM seeds averaged in normalised anomaly space (earlier experiments showed LSTM dramatically outperforms the Transformer here; mixing in a Transformer at equal weight *lowered* the ensemble val score, so v5 drops it).

**Target hardware.** Any CUDA GPU with â‰¥8 GB works. Training is light (~2M-param BiLSTM, 100 epochs/seed Ã— 6 seeds). On an A100/L4 the full ensemble takes ~30-40 min; on an RTX 4060 Ti it's ~80 min.

**Submission layout** (matches Shipd grader):
- reads from `dataset/public/{train,test}.csv`
- writes to `working/submission.csv`

---

## 1. Clone Repo

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!git clone https://{token}@github.com/Phuc182219/Weather-Translation-Challenge.git
%cd Weather-Translation-Challenge

In [ ]:
!git pull

## 2. GPU Check

In [ ]:
!nvidia-smi

## 3. Install Dependencies
Colab ships with torch/numpy/pandas already. We just make sure everything is up to date.

In [ ]:
!pip install -q -U torch numpy pandas

## 4. Set Up Sandbox Layout
The grader expects `dataset/public/{train,test}.csv` and writes to `working/submission.csv`. The repo already contains the public split under `public/` â€” we copy it into place.

In [ ]:
import os, shutil
os.makedirs('dataset/public', exist_ok=True)
for fn in ('train.csv', 'test.csv', 'sample_submission.csv'):
    src = os.path.join('public', fn)
    dst = os.path.join('dataset/public', fn)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
os.makedirs('working', exist_ok=True)
!ls -la dataset/public working

## 5. Explore Data

In [ ]:
import pandas as pd, numpy as np

train = pd.read_csv('dataset/public/train.csv')
test  = pd.read_csv('dataset/public/test.csv')

print(f'Train rows: {len(train):,}   cols: {len(train.columns)}')
print(f'Test  rows: {len(test):,}   cols: {len(test.columns)}')

# Station pairs â€” 12 in train, 14 in test (2 unseen combinations to test generalisation)
train_pairs = set(zip(train['source_city'], train['target_city']))
test_pairs  = set(zip(test['source_city'],  test['target_city']))
print(f'\nTrain pairs: {len(train_pairs)}   Test pairs: {len(test_pairs)}')
print(f'Pairs in TEST but not in TRAIN (unseen): {test_pairs - train_pairs}')
print(f'Pairs in TRAIN but not in TEST:          {train_pairs - test_pairs}')

print('\nPer-pair sample counts (test):')
print(test.groupby(['source_city','target_city']).size())

In [ ]:
# Per-station climate means (from train)
VARS = ('temp', 'dewpoint', 'wind_speed')
T = 72
stations = sorted(set(train['source_city']).union(train['target_city'])
                  .union(test['source_city']).union(test['target_city']))
print(f'Stations: {stations}')

def series_3d(df, role):
    cols = {v: [f'{role}_{v}_{i}' for i in range(T)] for v in VARS}
    return np.stack([df[cols[v]].values for v in VARS], axis=-1).astype(np.float32)

Xs_tr = series_3d(train, 'source')
Xt_tr = series_3d(train, 'target')

print('\nClimate mean per station (temp / dewpoint / wind_speed):')
for s in stations:
    ms = train['source_city'] == s
    mt = train['target_city'] == s
    src = Xs_tr[ms].reshape(-1, 3) if ms.any() else np.empty((0, 3))
    tgt = Xt_tr[mt].reshape(-1, 3) if mt.any() else np.empty((0, 3))
    comb = np.concatenate([src, tgt]) if len(src) + len(tgt) else np.array([[np.nan]*3])
    m = comb.mean(axis=0)
    print(f'  {s}: temp={m[0]:6.2f}  dp={m[1]:6.2f}  ws={m[2]:5.2f}    '
          f'(n_as_src={int(ms.sum())}, n_as_tgt={int(mt.sum())})')

In [ ]:
# Identity baseline: predict target = source. Gives a feel for how much
# transformation each pair actually requires.
print('Per-pair identity-baseline MSE (lower = source already close to target):')
for (s, t), g in train.groupby(['source_city','target_city']):
    src = series_3d(g, 'source')
    tgt = series_3d(g, 'target')
    mse = ((src - tgt) ** 2).mean()
    print(f'  {s} -> {t}: {mse:7.2f}   (d_temp={src[..., 0].mean()-tgt[..., 0].mean():+6.2f}, '
          f'd_dp={src[..., 1].mean()-tgt[..., 1].mean():+6.2f}, '
          f'd_ws={src[..., 2].mean()-tgt[..., 2].mean():+6.2f})')

print(f'\nOverall target variance (score reference): {Xt_tr.var():.3f}')

## 6. Run Solution
Switch between versions by changing `SOLUTION_VERSION`. The root `solution.py` is the current working copy; `blue/vN/solution.py` snapshots preserve each iteration.

In [ ]:
# === Choose which version ===
SOLUTION_VERSION = 'solution.py'             # root: latest working copy (v8)
# SOLUTION_VERSION = 'blue/v8/solution.py'   # v8: bigger BiLSTM + RH/VPD features + TTA + 7-model ensemble
# SOLUTION_VERSION = 'blue/v7/solution.py'   # v7: coupled pair-dropout ensemble (val 0.8411 diverse)
# SOLUTION_VERSION = 'blue/v6/solution.py'   # v6: emb-dropout + mixup + TCN             (val 0.8153, test 0.7092)
# SOLUTION_VERSION = 'blue/v5/solution.py'   # v5: climate-centered BiLSTM ensemble       (val 0.8355, test 0.7281)
# SOLUTION_VERSION = 'blue/v4/solution.py'   # v4: TFM + LSTM ensemble                    (val 0.790,  test 0.6765)
# SOLUTION_VERSION = 'blue/v3/solution.py'   # v3: 5-seed Transformer ensemble            (val 0.7414)
# SOLUTION_VERSION = 'blue/v2/solution.py'   # v2: variance-weighted loss + EMA (worse, kept for reference)
# SOLUTION_VERSION = 'blue/v1/solution.py'   # v1: single Transformer baseline            (val 0.71,   test 0.6032)

!python -u {SOLUTION_VERSION} 2>&1 | tee working/run.log

## 7. Validate Submission

In [ ]:
import pandas as pd, numpy as np

sub    = pd.read_csv('working/submission.csv')
sample = pd.read_csv('dataset/public/sample_submission.csv')
test   = pd.read_csv('dataset/public/test.csv')

print(f'Submission shape:    {sub.shape}   (expected ({len(test)}, 217))')
print(f'Sample shape:        {sample.shape}')
print(f'Columns match sample: {list(sub.columns) == list(sample.columns)}')
print(f'IDs match test:       {(sub["id"].values == test["id"].values).all()}')
print(f'Any NaNs:             {int(sub.isna().sum().sum())}')
print(f'Any infs:             {int(np.isinf(sub.select_dtypes(include="number").values).sum())}')

# Sanity-check physical ranges
num = sub.select_dtypes(include='number')
vars_ = {'temp': (-50, 55), 'dewpoint': (-55, 35), 'wind_speed': (0, 40)}
for v, (lo, hi) in vars_.items():
    cols = [c for c in num.columns if c.startswith(f'target_{v}_')]
    vals = num[cols].values
    print(f'  {v}: min={vals.min():7.2f}, max={vals.max():7.2f} (allowed [{lo}, {hi}])')

In [ ]:
# If your run also printed a validation score, that estimate and the real Shipd
# score differ because the validation split contains only pair combinations that
# are present in training. The Shipd test set includes 2 pair combinations that
# are *unseen* during training (~14% of test samples) which strictly lowers the
# final score. The climate-anomaly decomposition in v5 is specifically aimed at
# closing that gap.
print('Unseen test pair IDs (for sanity):')
train = pd.read_csv('dataset/public/train.csv')
train_pairs = set(zip(train['source_city'], train['target_city']))
unseen_mask = [(s, t) not in train_pairs for s, t in zip(test['source_city'], test['target_city'])]
print(f'  unseen-pair test rows: {sum(unseen_mask):,} / {len(test):,} '
      f'({100*sum(unseen_mask)/len(test):.1f}%)')
for (s, t), n in test[unseen_mask].groupby(['source_city','target_city']).size().items():
    print(f'    {s} -> {t}: {n}')

## 8. Download Submission

In [ ]:
from google.colab import files
import shutil

# Copy working/submission.csv to a renamed file first so the download
# keeps a unique name across different Shipd challenges.
#
# On Shipd, ranking is by Script Run Score (= score of the CSV generated
# when Shipd runs solution.py). The CSV Upload is just proof-of-work that
# your script actually produced that output. So this downloaded file
# should exactly match what Shipd will regenerate when it runs the script.
dl_name = 'Weather Translation Challenge submission.csv'
shutil.copy('working/submission.csv', dl_name)
files.download(dl_name)

## 9. (Optional) Push results back to GitHub
If the run produced a new submission / updated solution you want to keep, commit and push from the notebook.

In [ ]:
# Configure a git identity (safe to re-run)
!git config user.email "phucnlhse182219@gmail.com"
!git config user.name  "Phuc182219"

# Snapshot this run as a new blue/vN (bump N as appropriate)
# import shutil, os
# V = 'v5_colab'
# os.makedirs(f'blue/{V}', exist_ok=True)
# shutil.copy('solution.py',              f'blue/{V}/solution.py')
# shutil.copy('working/submission.csv',   f'blue/{V}/submission.csv')
# !git add blue/{V}/ working/submission.csv
# !git commit -m "Colab run: {V}"
# !git push